# TRÍCH XUẤT ĐẶC TRƯNG HÀNG KHÔNG

**Mục tiêu:** Trích xuất 19 đặc trưng cốt lõi từ dữ liệu Silver Layer để phục vụ cho mô hình dự đoán trễ chuyến dây chuyền.

In [44]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Khai báo hằng số hệ thống
AIRPORTS = ("SGN", "HAN", "DAD")
WIDE_BODY_TYPES = {"A359", "B789", "B747", "A330", "B77X", "B78X", "A333", "B77W", "B788"}
LCC_HINTS = ("vietjet", "bamboo", "vietravel", "sun phuquoc", "pacific airlines")
FSC_HINTS = ("vietnam airlines",)
DISRUPTION_STATUS_HINTS = ("cancel", "divert", "return", "emergency")

pd.options.mode.chained_assignment = None  # Tắt cảnh báo chain assignment để output sạch sẽ

## 1. Định nghĩa Engine xử lý cốt lõi
Cấu trúc này sẽ đảm bảo tính toàn vẹn của dữ liệu và xử lý các lỗi rỗng (NaN) một cách triệt để.

In [45]:
class AeroDelayFeatureEngineer:
    """Feature engineering pipeline for Silver-layer AeroDelay flight data."""

    def __init__(self, project_root=None):
        if project_root is None:
            self.project_root = Path(os.getcwd()).resolve().parent
        else:
            self.project_root = Path(project_root).resolve()
            
        self.silver_path = self.project_root / "Data crawl" / "Silver_layer"
        self.arrival_path = self.silver_path / "Arrival"
        self.departure_path = self.silver_path / "Departure"
        self.audit_path = self.silver_path / "Audit"
        self.gold_path = self.project_root / "Data crawl" / "Gold_layer"
        self.gold_arrival_path = self.gold_path / "Arrival"
        self.gold_departure_path = self.gold_path / "Departure"
        self.gold_features_path = self.gold_path / "Features"
        self.gold_audit_path = self.gold_path / "Audit"
        
        for path in (self.gold_arrival_path, self.gold_departure_path, self.gold_features_path, self.gold_audit_path):
            os.makedirs(path, exist_ok=True)
            
        self.master_output_path = self.gold_features_path / "master_aero_features_gold.csv"
        self.data_leakage_report_path = self.gold_audit_path / "temporal_data_leakage_report.csv"
        self.imputation_metrics_path = self.gold_audit_path / "imputation_metrics_log.csv"
        self.feature_summary_stats_path = self.gold_audit_path / "feature_summary_stats.csv"

        self.arrivals = None
        self.departures = None
        self.features = None
        self.arrival_features = None
        self.master_features = None
        self.audit_missing_expected = 0
        self.source_file_paths = {}
        self.source_file_columns = {}

    @staticmethod
    def _read_csv(path):
        return pd.read_csv(path, low_memory=False)

    @staticmethod
    def _clean_string(series):
        return series.astype("string").str.strip().replace({"": pd.NA, "N/A": pd.NA, "nan": pd.NA})

    @staticmethod
    def _bool_series(series, default=False):
        if series is None:
            return pd.Series(dtype=bool)
        mapped = (series.astype("string").str.strip().str.lower()
                  .map({"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}))
        return mapped.astype("boolean").fillna(default).astype(bool)

    @staticmethod
    def _normalize_flight_no(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper()
        if text.startswith("BL"):
            return "VN" + text[2:]
        return text

    @staticmethod
    def _minutes(delta):
        return delta.dt.total_seconds().div(60)

    @staticmethod
    def _feature_output_columns():
        return [
            "Record_Type", "Origin", "Destination", "Crawl_Date", "Flight_No", "Airline", "Airline_Type",
            "Scheduled_Time", "Actual_Time", "Imputed_Actual_Departure", "Departure_Delay", "Scheduled_Tail",
            "Actual_Tail", "Aircraft_Type", "Category", "Terminal", "Departure_Runway", "Arrival_Runway",
            "Status", "Tail_Sequence_Day", "Is_First_Flight", "Arrival_Delay", "Accumulated_Delay",
            "Turnaround_Buffer", "Tail_Stagnation_Duration", "Is_Diverted_Prev", "Airport_Load_Factor",
            "Number_of_Flights_in_Last_Hour", "Is_Airport_Congested", "Flight_Density_Disruption",
            "Previous_Station_Disruption", "Is_Parallel_Usage", "Runway_Swap_Event", "Is_Wide_Body",
            "Time_of_Day", "Peak_Hour_Indicator", "Is_Special_Days", "Is_Imputed_Link", "Missing_Link_Type",
            "Prev_Missing_Link_Type", "Exclude_From_Propagation_Training", "Match_Status", "Data_Completeness",
            "Prev_Actual_Arrival", "Imputed_Actual_Arrival", "Previous_Station", "Standard_Turnaround"
        ]

    def _load_mode_files(self, folder, mode):
        frames = []
        for airport in AIRPORTS:
            candidates = sorted(folder.glob(f"{airport.lower()}_flights_{mode}_*.csv"))
            if not candidates:
                raise FileNotFoundError(f"Missing {mode} silver file for {airport} in {folder}")
            path = candidates[0]
            df = self._read_csv(path)
            self.source_file_paths[(mode, airport)] = path
            self.source_file_columns[(mode, airport)] = list(df.columns)
            df["Airport"] = airport
            df["Source_File"] = path.name
            df["Source_Row_Index"] = np.arange(len(df))
            frames.append(df)
        return pd.concat(frames, ignore_index=True, sort=False)

    def _apply_deduplicate_audit(self, df, mode):
        path = self.audit_path / "audit_deduplicate_decisions.csv"
        if not path.exists(): return df
        audit = self._read_csv(path)
        audit = audit[audit["mode"].astype("string").str.lower().eq(mode)]
        if audit.empty: return df
        drop_keys = set(zip(audit["airport"].astype(str), audit["row_index_dropped"].astype(int)))
        key = list(zip(df["Airport"].astype(str), df["Source_Row_Index"].astype(int)))
        return df.loc[[item not in drop_keys for item in key]].copy()

    def _apply_same_origin_actions(self, arrivals):
        path = self.audit_path / "audit_same_origin_actions.csv"
        if not path.exists(): return arrivals
        audit = self._read_csv(path)
        drop_actions = {"drop_unmatched_same_origin", "drop_unmatched_flights"}
        audit = audit[audit["Action"].isin(drop_actions)]
        drop_keys = set(zip(audit["Airport"].astype(str), audit["Row_Index"].astype(int)))
        key = list(zip(arrivals["Airport"].astype(str), arrivals["Source_Row_Index"].astype(int)))
        return arrivals.loc[[item not in drop_keys for item in key]].copy()

    def _standardize_common_fields(self, df, mode):
        df = df.copy()
        for col in ("Flight_No", "Airline", "IATA", "Aircraft_Type", "Status", "Category"):
            if col in df: df[col] = self._clean_string(df[col])
        df["Flight_No"] = df["Flight_No"].map(self._normalize_flight_no).astype("string")
        df["Scheduled_Time"] = pd.to_datetime(df.get("Scheduled_Time"), errors="coerce")
        df["Actual_Time"] = pd.to_datetime(df.get("Actual_Time"), errors="coerce")
        df["Crawl_Date"] = pd.to_datetime(df.get("Crawl_Date"), errors="coerce").dt.date
        df["Mode"] = mode

        if mode == "departure":
            df["Tail_Number"] = self._clean_string(df.get("Matched_Actual_Tail", pd.Series(index=df.index, dtype="object"))).fillna(self._clean_string(df.get("Scheduled_Tail", pd.Series(index=df.index, dtype="object"))))
            df["Origin"] = df["Airport"]
            df["Destination_IATA"] = df["IATA"]
            df["Origin_IATA"] = df["Airport"]
            df["Event_Time"] = df["Scheduled_Time"].fillna(df["Actual_Time"])
            df["Actual_Event_Time"] = df["Actual_Time"]
            df["Scheduled_Event_Time"] = df["Scheduled_Time"].fillna(df["Actual_Time"])
            df["Departure_Delay"] = self._minutes(df["Actual_Time"] - df["Scheduled_Time"]).fillna(0)
        else:
            df["Tail_Number"] = self._clean_string(df.get("Actual_Tail", pd.Series(index=df.index, dtype="object")))
            df["Destination"] = df["Airport"]
            df["Origin_IATA"] = df["IATA"]
            df["Destination_IATA"] = df["Airport"]
            df["Event_Time"] = df["Actual_Time"]
            df["Actual_Event_Time"] = df["Actual_Time"]
            df["Scheduled_Event_Time"] = df["Scheduled_Time"].fillna(df["Actual_Time"])
            df["Arrival_Delay_This_Leg"] = self._minutes(df["Actual_Time"] - df["Scheduled_Time"]).fillna(0)

        if "Exclude_From_Propagation_Training" in df:
            df["Exclude_From_Propagation_Training"] = self._bool_series(df["Exclude_From_Propagation_Training"])
        else:
            df["Exclude_From_Propagation_Training"] = False
        non_passenger = df.get("Category", pd.Series(pd.NA, index=df.index)).astype("string").str.lower().ne("passenger")
        df.loc[non_passenger.fillna(True), "Exclude_From_Propagation_Training"] = True
        return df

    def load_and_clean_base_data(self):
        print("[*] 1/7: Đang làm sạch dữ liệu nền từ các file log Audit...")
        arrivals = self._load_mode_files(self.arrival_path, "arrival")
        departures = self._load_mode_files(self.departure_path, "departure")

        arrivals = self._apply_deduplicate_audit(arrivals, "arrival")
        departures = self._apply_deduplicate_audit(departures, "departure")
        arrivals = self._apply_same_origin_actions(arrivals)

        arrivals = self._standardize_common_fields(arrivals, "arrival")
        departures = self._standardize_common_fields(departures, "departure")

        arrivals = arrivals[arrivals["Actual_Time"].notna() & arrivals["Tail_Number"].notna()].copy()
        departures = departures[departures["Scheduled_Time"].notna() & departures["Event_Time"].notna() & departures["Tail_Number"].notna()].copy()

        self.arrivals = arrivals.reset_index(drop=True)
        self.departures = departures.reset_index(drop=True)
        print(f"[V] Đã nạp {len(self.departures)} chuyến cất cánh và {len(self.arrivals)} chuyến hạ cánh.")

    def _load_missing_audits(self):
        arr_path = self.audit_path / "audit_arrival_without_departure.csv"
        dep_path = self.audit_path / "audit_departure_without_arrival.csv"
        arr_missing = self._read_csv(arr_path) if arr_path.exists() else pd.DataFrame()
        dep_missing = self._read_csv(dep_path) if dep_path.exists() else pd.DataFrame()
        self.audit_missing_expected = len(arr_missing) + len(dep_missing)
        return arr_missing, dep_missing

    @staticmethod
    def _audit_key(df, airport_col, time_col):
        flight = df["Flight_No"].map(AeroDelayFeatureEngineer._normalize_flight_no).astype("string")
        airport = df[airport_col].astype("string").str.upper()
        time = pd.to_datetime(df[time_col], errors="coerce").dt.floor("min").astype("string")
        return airport.fillna("") + "|" + flight.fillna("") + "|" + time.fillna("")

    def _flag_missing_links(self, arrivals, departures):
        arr_missing, dep_missing = self._load_missing_audits()
        arrivals = arrivals.copy()
        departures = departures.copy()
        arrivals["Is_Imputed_Link"] = 0
        departures["Is_Imputed_Link"] = 0
        arrivals["Missing_Link_Type"] = pd.NA
        departures["Missing_Link_Type"] = pd.NA

        if not arr_missing.empty:
            arr_keys = set(self._audit_key(arr_missing, "Destination", "Arr_Event_Time"))
            current_keys = self._audit_key(arrivals, "Airport", "Actual_Time")
            mask = current_keys.isin(arr_keys)
            arrivals.loc[mask, "Is_Imputed_Link"] = 1
            arrivals.loc[mask, "Missing_Link_Type"] = "arrival_without_departure"

        if not dep_missing.empty:
            dep_keys = set(self._audit_key(dep_missing, "Origin", "Dep_Event_Time"))
            current_keys = self._audit_key(departures, "Airport", "Actual_Time")
            mask = current_keys.isin(dep_keys)
            departures.loc[mask, "Is_Imputed_Link"] = 1
            departures.loc[mask, "Missing_Link_Type"] = "departure_without_arrival"
        return arrivals, departures

    def build_tail_rotation_features(self):
        print("[*] 2/7: Đang xây dựng chuỗi xoay vòng tàu bay (Tail Rotation)...")
        arrivals, departures = self._flag_missing_links(self.arrivals, self.departures)

        clean_arrivals = arrivals[arrivals["Is_Imputed_Link"].eq(0)].copy()
        clean_departures = departures[departures["Is_Imputed_Link"].eq(0)].copy()
        clean_turns = pd.merge_asof(
            clean_departures.sort_values("Scheduled_Time"),
            clean_arrivals.sort_values("Actual_Time")[["Tail_Number", "Airport", "Actual_Time"]]
            .rename(columns={"Airport": "Prev_Arrival_Airport", "Actual_Time": "Prev_Actual_Arrival"}),
            left_on="Scheduled_Time", right_on="Prev_Actual_Arrival", by="Tail_Number", direction="backward", allow_exact_matches=False
        )
        clean_turns = clean_turns[clean_turns["Prev_Actual_Arrival"].notna()].copy()
        clean_turns["Actual_Turnaround"] = self._minutes(clean_turns["Actual_Time"] - clean_turns["Prev_Actual_Arrival"])
        
        baseline = clean_turns[clean_turns["Actual_Turnaround"].between(20, 720)].groupby(["Aircraft_Type", "Airport"], dropna=False)["Actual_Turnaround"].median().rename("Median_Turnaround_Time").reset_index()
        global_turnaround = float(baseline["Median_Turnaround_Time"].median()) if len(baseline) else 60.0

        arrivals = arrivals.merge(baseline, on=["Aircraft_Type", "Airport"], how="left")
        arrivals["Median_Turnaround_Time"] = arrivals["Median_Turnaround_Time"].fillna(global_turnaround)
        arrivals["Observed_Actual_Arrival"] = arrivals["Actual_Time"]
        arrivals.loc[arrivals["Is_Imputed_Link"].eq(1), "Arrival_Delay_This_Leg"] = 0

        prev_arrivals = arrivals[[
            "Tail_Number", "Airport", "Origin_IATA", "Actual_Time", "Status", "Arrival_Delay_This_Leg",
            "Median_Turnaround_Time", "Is_Imputed_Link", "Missing_Link_Type", "Is_Return_Emergency"
        ]].rename(columns={
            "Airport": "Prev_Arrival_Airport", "Origin_IATA": "Previous_Station", "Actual_Time": "Prev_Actual_Arrival",
            "Status": "Prev_Arrival_Status", "Arrival_Delay_This_Leg": "Arrival_Delay",
            "Median_Turnaround_Time": "Standard_Turnaround", "Is_Imputed_Link": "Prev_Arrival_Imputed_Link",
            "Missing_Link_Type": "Prev_Missing_Link_Type"
        })
        
        features = pd.merge_asof(
            departures.sort_values("Scheduled_Time"), prev_arrivals.sort_values("Prev_Actual_Arrival"),
            left_on="Scheduled_Time", right_on="Prev_Actual_Arrival", by="Tail_Number", direction="backward", allow_exact_matches=False
        )

        features["Observed_Actual_Departure"] = features["Actual_Time"]
        features["Observed_Prev_Actual_Arrival"] = features["Prev_Actual_Arrival"]
        features["Standard_Turnaround"] = features["Standard_Turnaround"].fillna(global_turnaround)

        imputed_prev_arrival = features["Prev_Arrival_Imputed_Link"].astype("boolean").fillna(False).astype(bool)
        features["Imputed_Actual_Arrival"] = pd.NaT
        features.loc[imputed_prev_arrival, "Imputed_Actual_Arrival"] = features.loc[imputed_prev_arrival, "Scheduled_Time"] - pd.to_timedelta(features.loc[imputed_prev_arrival, "Standard_Turnaround"], unit="m")
        features.loc[imputed_prev_arrival, "Prev_Actual_Arrival"] = features.loc[imputed_prev_arrival, "Imputed_Actual_Arrival"]
        features.loc[imputed_prev_arrival, "Arrival_Delay"] = 0

        imputed_departure = features["Is_Imputed_Link"].eq(1)
        can_impute_departure = imputed_departure & features["Prev_Actual_Arrival"].notna()
        features["Imputed_Actual_Departure"] = pd.NaT
        features.loc[can_impute_departure, "Imputed_Actual_Departure"] = features.loc[can_impute_departure, "Prev_Actual_Arrival"] + pd.to_timedelta(features.loc[can_impute_departure, "Standard_Turnaround"], unit="m")
        features.loc[can_impute_departure, "Actual_Time"] = features.loc[can_impute_departure, "Imputed_Actual_Departure"]
        features.loc[can_impute_departure, "Departure_Delay"] = self._minutes(features.loc[can_impute_departure, "Actual_Time"] - features.loc[can_impute_departure, "Scheduled_Time"]).fillna(0)

        features["Flight_Date"] = features["Scheduled_Time"].dt.date
        features["Tail_Sequence_Day"] = features.sort_values(["Tail_Number", "Scheduled_Time"]).groupby(["Tail_Number", "Flight_Date"], dropna=False).cumcount().add(1).astype(int)
        features["Is_First_Flight"] = (features["Tail_Sequence_Day"] == 1).astype(int)
        features["Arrival_Delay"] = features["Arrival_Delay"].fillna(0)

        features = features.sort_values(["Tail_Number", "Scheduled_Time"]).copy()
        features["Accumulated_Delay"] = features.groupby(["Tail_Number", "Flight_Date"], dropna=False)["Arrival_Delay"].cumsum().sub(features["Arrival_Delay"]).clip(lower=0)
        features["Turnaround_Buffer"] = self._minutes(features["Scheduled_Time"] - features["Prev_Actual_Arrival"])
        features["Actual_Turnaround"] = self._minutes(features["Actual_Time"] - features["Prev_Actual_Arrival"])
        features["Standard_Turnaround"] = features["Standard_Turnaround"].fillna(global_turnaround)
        features["Tail_Stagnation_Duration"] = (features["Actual_Turnaround"] - features["Standard_Turnaround"]).fillna(0)

        prev_status = features["Prev_Arrival_Status"].astype("string").str.lower().fillna("")
        return_emergency = self._bool_series(features.get("Is_Return_Emergency", pd.Series(False, index=features.index)))
        features["Is_Diverted_Prev"] = (prev_status.str.contains("divert|return|emergency", regex=True) | return_emergency).astype(int)
        features["Is_Imputed_Link"] = (features["Is_Imputed_Link"].astype("boolean").fillna(False).astype(bool) | features["Prev_Arrival_Imputed_Link"].astype("boolean").fillna(False).astype(bool)).astype(int)

        self.arrivals = arrivals
        updated_departures = departures.merge(features[["Source_File", "Source_Row_Index", "Departure_Delay", "Imputed_Actual_Departure"]], on=["Source_File", "Source_Row_Index"], how="left", suffixes=("", "_Feature"))
        has_imputed = updated_departures["Imputed_Actual_Departure"].notna()
        updated_departures.loc[has_imputed, "Actual_Time"] = updated_departures.loc[has_imputed, "Imputed_Actual_Departure"]
        updated_departures.loc[has_imputed, "Actual_Event_Time"] = updated_departures.loc[has_imputed, "Imputed_Actual_Departure"]
        updated_departures.loc[has_imputed, "Departure_Delay"] = updated_departures.loc[has_imputed, "Departure_Delay_Feature"]
        self.departures = updated_departures
        self.features = features.reset_index(drop=True)
        print("[V] Xử lý thành công dữ liệu xoay vòng.")

    def _build_airport_event_table(self):
        arr_events = self.arrivals.assign(Airport_Event=self.arrivals["Airport"], Scheduled_Event_Time=self.arrivals["Scheduled_Event_Time"], Actual_Event_Time=self.arrivals["Actual_Event_Time"], Event_Status=self.arrivals["Status"])[["Airport_Event", "Scheduled_Event_Time", "Actual_Event_Time", "Event_Status"]]
        dep_events = self.departures.assign(Airport_Event=self.departures["Airport"], Scheduled_Event_Time=self.departures["Scheduled_Event_Time"], Actual_Event_Time=self.departures["Actual_Event_Time"], Event_Status=self.departures["Status"])[["Airport_Event", "Scheduled_Event_Time", "Actual_Event_Time", "Event_Status"]]
        events = pd.concat([arr_events, dep_events], ignore_index=True)
        events = events[events["Scheduled_Event_Time"].notna()].copy()
        events["Is_Disrupted_Event"] = events["Event_Status"].astype("string").str.lower().fillna("").str.contains("|".join(DISRUPTION_STATUS_HINTS), regex=True)
        return events

    @staticmethod
    def _count_between(sorted_values, starts, ends, side_right="right"):
        left = np.searchsorted(sorted_values, starts, side="left")
        right = np.searchsorted(sorted_values, ends, side=side_right)
        return right - left

    def build_airport_infrastructure_features(self):
        print("[*] 3/7: Đang trích xuất đặc trưng Hạ tầng và Sự cố (Rolling Window)...")
        features = self.features.copy()
        events = self._build_airport_event_table()
        features["Airport_Load_Factor"] = 0
        features["Number_of_Flights_in_Last_Hour"] = 0
        features["Flight_Density_Disruption"] = 0.0

        for airport, idx in features.groupby("Airport").groups.items():
            idx = np.array(list(idx))
            t = features.loc[idx, "Scheduled_Time"].values.astype("datetime64[ns]")
            ev = events[events["Airport_Event"].eq(airport)]
            scheduled = np.sort(ev["Scheduled_Event_Time"].dropna().values.astype("datetime64[ns]"))
            actual = np.sort(ev["Actual_Event_Time"].dropna().values.astype("datetime64[ns]"))
            disrupted = np.sort(ev.loc[ev["Is_Disrupted_Event"], "Scheduled_Event_Time"].dropna().values.astype("datetime64[ns]"))
            
            features.loc[idx, "Airport_Load_Factor"] = self._count_between(scheduled, t - np.timedelta64(30, "m"), t + np.timedelta64(30, "m"))
            features.loc[idx, "Number_of_Flights_in_Last_Hour"] = self._count_between(actual, t - np.timedelta64(60, "m"), t, side_right="left")
            denom = self._count_between(scheduled, t - np.timedelta64(60, "m"), t, side_right="left")
            numer = self._count_between(disrupted, t - np.timedelta64(60, "m"), t, side_right="left")
            features.loc[idx, "Flight_Density_Disruption"] = np.divide(numer, denom, out=np.zeros_like(numer, dtype=float), where=denom > 0)

        congestion = features.groupby("Airport")["Number_of_Flights_in_Last_Hour"].quantile(0.85)
        load = features.groupby("Airport")["Airport_Load_Factor"].quantile(0.85)
        features["Is_Airport_Congested"] = (features["Number_of_Flights_in_Last_Hour"] > features["Airport"].map(congestion).fillna(np.inf)).astype(int)
        features["Is_Parallel_Usage"] = (features["Airport_Load_Factor"] >= features["Airport"].map(load).fillna(np.inf)).astype(int)

        current_disruption_map = {airport: group[["Scheduled_Time", "Flight_Density_Disruption"]].sort_values("Scheduled_Time") for airport, group in features.groupby("Airport")}
        features["Previous_Station_Disruption"] = 0.0
        for airport, source in current_disruption_map.items():
            idx = features.index[features["Previous_Station"].eq(airport)]
            if len(idx) == 0: continue
            lookup = features.loc[idx, ["Scheduled_Time"]].copy()
            lookup["__row_id"] = lookup.index
            prev = pd.merge_asof(lookup.sort_values("Scheduled_Time"), source, on="Scheduled_Time", direction="backward")
            features.loc[prev["__row_id"].values, "Previous_Station_Disruption"] = prev["Flight_Density_Disruption"].fillna(0).values

        delay_proxy = features["Departure_Delay"].clip(lower=0)
        airport_delay_median = delay_proxy.groupby(features["Airport"]).transform("median").clip(lower=1)
        features["Runway_Swap_Event"] = ((delay_proxy > airport_delay_median * 2) & (delay_proxy > 15)).astype(int)
        self.features = features
        print("[V] Đã xử lý xong biến hạ tầng (np.searchsorted siêu tốc).")

    def build_operational_and_holiday_features(self):
        print("[*] 4/7: Đang lập lịch đặc trưng Vận hành và Lễ Tết...")
        df = self.features.copy()
        df["Record_Type"] = "Departure"
        df["Is_Wide_Body"] = df["Aircraft_Type"].astype("string").str.upper().isin(WIDE_BODY_TYPES).astype(int)

        airline = df["Airline"].astype("string").str.lower().fillna("")
        df["Airline_Type"] = np.select(
            [airline.str.contains("|".join(LCC_HINTS), regex=True), airline.str.contains("|".join(FSC_HINTS), regex=True)],
            ["LCC", "FSC"], default="Other"
        )

        hour = df["Scheduled_Time"].dt.hour
        df["Time_of_Day"] = np.select(
            [hour.between(0, 5), hour.between(6, 11), hour.between(12, 17), hour.between(18, 23)],
            ["Early_Morning", "Morning", "Afternoon", "Night"], default="Unknown"
        )
        minute = df["Scheduled_Time"].dt.hour.mul(60).add(df["Scheduled_Time"].dt.minute)
        df["Peak_Hour_Indicator"] = (minute.between(7 * 60, 9 * 60) | minute.between(16 * 60 + 30, 19 * 60 + 30)).astype(int)

        date = df["Scheduled_Time"].dt.date
        month = df["Scheduled_Time"].dt.month
        fixed_holidays = (month.isin([6, 7]) | date.isin(pd.to_datetime(["2025-12-24", "2025-12-25", "2025-12-31", "2026-01-01", "2026-02-14", "2026-02-15", "2026-02-16", "2026-02-17", "2026-02-18", "2026-02-19", "2026-02-20", "2026-04-26", "2026-04-30", "2026-05-01", "2026-09-02"]).date))
        df["Is_Special_Days"] = fixed_holidays.astype(int)
        self.features = df

    def build_arrival_feature_rows(self):
        print("[*] 5/7: Đang map feature tích lũy vào các chuyến hạ cánh...")
        arrivals = self.arrivals.copy()
        arrivals["Record_Type"] = "Arrival"
        arrivals["Flight_Date"] = arrivals["Actual_Time"].dt.date
        arrivals["Tail_Sequence_Day"] = arrivals.sort_values(["Tail_Number", "Actual_Time"]).groupby(["Tail_Number", "Flight_Date"], dropna=False).cumcount().add(1).astype(int)
        arrivals["Is_First_Flight"] = (arrivals["Tail_Sequence_Day"] == 1).astype(int)
        arrivals["Arrival_Delay"] = arrivals["Arrival_Delay_This_Leg"].fillna(0)
        arrivals = arrivals.sort_values(["Tail_Number", "Actual_Time"]).copy()
        arrivals["Accumulated_Delay"] = arrivals.groupby(["Tail_Number", "Flight_Date"], dropna=False)["Arrival_Delay"].cumsum().sub(arrivals["Arrival_Delay"]).clip(lower=0)
        
        arrivals["Standard_Turnaround"] = arrivals["Median_Turnaround_Time"]
        arrivals["Observed_Prev_Actual_Arrival"] = arrivals["Observed_Actual_Arrival"]
        arrivals["Imputed_Actual_Arrival"] = pd.NaT
        arrivals["Prev_Actual_Arrival"] = arrivals["Actual_Time"]
        arrivals["Previous_Station"] = arrivals["Origin_IATA"]
        for col in ["Observed_Actual_Departure", "Imputed_Actual_Departure", "Departure_Delay", "Turnaround_Buffer"]:
            arrivals[col] = np.nan
        arrivals["Tail_Stagnation_Duration"] = 0.0
        arrivals["Is_Diverted_Prev"] = self._bool_series(arrivals.get("Is_Return_Emergency", pd.Series(False, index=arrivals.index))).astype(int)
        arrivals["Prev_Missing_Link_Type"] = pd.NA

        events = self._build_airport_event_table()
        arrivals["Airport_Load_Factor"] = 0
        arrivals["Number_of_Flights_in_Last_Hour"] = 0
        arrivals["Flight_Density_Disruption"] = 0.0
        for airport, idx in arrivals.groupby("Airport").groups.items():
            idx = np.array(list(idx))
            t = arrivals.loc[idx, "Actual_Time"].values.astype("datetime64[ns]")
            ev = events[events["Airport_Event"].eq(airport)]
            scheduled = np.sort(ev["Scheduled_Event_Time"].dropna().values.astype("datetime64[ns]"))
            actual = np.sort(ev["Actual_Event_Time"].dropna().values.astype("datetime64[ns]"))
            disrupted = np.sort(ev.loc[ev["Is_Disrupted_Event"], "Scheduled_Event_Time"].dropna().values.astype("datetime64[ns]"))
            arrivals.loc[idx, "Airport_Load_Factor"] = self._count_between(scheduled, t - np.timedelta64(30, "m"), t + np.timedelta64(30, "m"))
            arrivals.loc[idx, "Number_of_Flights_in_Last_Hour"] = self._count_between(actual, t - np.timedelta64(60, "m"), t, side_right="left")
            denom = self._count_between(scheduled, t - np.timedelta64(60, "m"), t, side_right="left")
            numer = self._count_between(disrupted, t - np.timedelta64(60, "m"), t, side_right="left")
            arrivals.loc[idx, "Flight_Density_Disruption"] = np.divide(numer, denom, out=np.zeros_like(numer, dtype=float), where=denom > 0)

        congestion = arrivals.groupby("Airport")["Number_of_Flights_in_Last_Hour"].quantile(0.85)
        load = arrivals.groupby("Airport")["Airport_Load_Factor"].quantile(0.85)
        arrivals["Is_Airport_Congested"] = (arrivals["Number_of_Flights_in_Last_Hour"] > arrivals["Airport"].map(congestion).fillna(np.inf)).astype(int)
        arrivals["Is_Parallel_Usage"] = (arrivals["Airport_Load_Factor"] >= arrivals["Airport"].map(load).fillna(np.inf)).astype(int)
        arrivals["Previous_Station_Disruption"] = 0.0
        arrivals["Runway_Swap_Event"] = 0
        arrivals["Is_Wide_Body"] = arrivals["Aircraft_Type"].astype("string").str.upper().isin(WIDE_BODY_TYPES).astype(int)
        airline = arrivals["Airline"].astype("string").str.lower().fillna("")
        arrivals["Airline_Type"] = np.select([airline.str.contains("|".join(LCC_HINTS), regex=True), airline.str.contains("|".join(FSC_HINTS), regex=True)], ["LCC", "FSC"], default="Other")
        hour = arrivals["Actual_Time"].dt.hour
        arrivals["Time_of_Day"] = np.select([hour.between(0, 5), hour.between(6, 11), hour.between(12, 17), hour.between(18, 23)], ["Early_Morning", "Morning", "Afternoon", "Night"], default="Unknown")
        minute = arrivals["Actual_Time"].dt.hour.mul(60).add(arrivals["Actual_Time"].dt.minute)
        arrivals["Peak_Hour_Indicator"] = (minute.between(7 * 60, 9 * 60) | minute.between(16 * 60 + 30, 19 * 60 + 30)).astype(int)
        date = arrivals["Actual_Time"].dt.date
        month = arrivals["Actual_Time"].dt.month
        arrivals["Is_Special_Days"] = (month.isin([6, 7]) | date.isin(pd.to_datetime(["2025-12-24", "2025-12-25", "2025-12-31", "2026-01-01", "2026-02-14", "2026-02-15", "2026-02-16", "2026-02-17", "2026-02-18", "2026-02-19", "2026-02-20", "2026-04-26", "2026-04-30", "2026-05-01", "2026-09-02"]).date)).astype(int)
        self.arrival_features = arrivals

    def evaluate_features_quality(self):
        print("[*] 6/7: Đang kiểm định chất lượng (Data Leakage & Imputation)...")
        df = self.features.copy()
        violations = df.index[df["Prev_Actual_Arrival"].notna() & (df["Scheduled_Time"] <= df["Prev_Actual_Arrival"])].tolist()
        if violations:
            print(f"[!] CẢNH BÁO: Phát hiện {len(violations)} dòng vi phạm dòng thời gian.")
        else:
            print("[V] Xác thực Temporal Leakage thành công: 0 vi phạm.")

    def _build_master_feature_table(self):
        output_columns = self._feature_output_columns()
        for frame in (self.features, self.arrival_features):
            for col in output_columns:
                if col not in frame.columns: frame[col] = pd.NA
        master = pd.concat([self.features[output_columns], self.arrival_features[output_columns]], ignore_index=True)
        master["Sort_Time"] = pd.to_datetime(master["Scheduled_Time"], errors="coerce").fillna(pd.to_datetime(master["Actual_Time"], errors="coerce"))
        self.master_features = master.sort_values(["Sort_Time", "Record_Type", "Flight_No"]).drop(columns=["Sort_Time"]).reset_index(drop=True)
        return self.master_features

    def _prepare_gold_file(self, mode, airport, feature_rows, extra_cols):
        path = self.source_file_paths[(mode, airport)]
        silver_cols = self.source_file_columns[(mode, airport)].copy()
        source = self._read_csv(path)
        source["Source_File"] = path.name
        source["Source_Row_Index"] = np.arange(len(source))
        additions = feature_rows[["Source_File", "Source_Row_Index"] + [c for c in extra_cols if c in feature_rows.columns]].drop_duplicates(["Source_File", "Source_Row_Index"], keep="last")
        gold = source.merge(additions, on=["Source_File", "Source_Row_Index"], how="left")
        if mode == "arrival" and "Scheduled_Time" in silver_cols:
            silver_cols.remove("Scheduled_Time")
            gold = gold.drop(columns=["Scheduled_Time"], errors="ignore")
        allowed = silver_cols + [c for c in extra_cols if c not in silver_cols and not (mode == "arrival" and c == "Scheduled_Time")]
        return gold[[c for c in allowed if c in gold.columns]]

    def export_gold_outputs(self):
        print("[*] 7/7: Đang xuất bản dữ liệu sạch sang Gold Layer...")
        master = self._build_master_feature_table()
        master.to_csv(self.master_output_path, index=False)
        
        common_features = ["Record_Type", "Airline_Type", "Tail_Sequence_Day", "Is_First_Flight", "Arrival_Delay", "Accumulated_Delay", "Turnaround_Buffer", "Tail_Stagnation_Duration", "Is_Diverted_Prev", "Airport_Load_Factor", "Number_of_Flights_in_Last_Hour", "Is_Airport_Congested", "Flight_Density_Disruption", "Previous_Station_Disruption", "Is_Parallel_Usage", "Runway_Swap_Event", "Is_Wide_Body", "Time_of_Day", "Peak_Hour_Indicator", "Is_Special_Days", "Is_Imputed_Link", "Missing_Link_Type", "Prev_Missing_Link_Type", "Prev_Actual_Arrival", "Imputed_Actual_Arrival", "Previous_Station", "Standard_Turnaround"]
        
        for airport in AIRPORTS:
            dep_gold = self._prepare_gold_file("departure", airport, self.features, common_features + ["Origin", "Imputed_Actual_Departure", "Departure_Delay"])
            arr_gold = self._prepare_gold_file("arrival", airport, self.arrival_features, common_features + ["Destination"])
            dep_gold.to_csv(self.gold_departure_path / f"{airport.lower()}_flights_departure_gold_layer.csv", index=False)
            arr_gold.to_csv(self.gold_arrival_path / f"{airport.lower()}_flights_arrival_gold_layer.csv", index=False)
            
        print(f"[V] TIẾN TRÌNH HOÀN TẤT! Dữ liệu Master được lưu tại: {self.master_output_path}")


## 2. Khởi chạy Pipeline Từng Bước
Thực thi từng phương thức một cách an toàn mà không sợ lỗi ghi đè dữ liệu.

In [46]:
# Khởi tạo bộ máy Engineer
engineer = AeroDelayFeatureEngineer()

# BƯỚC 1: Đọc và tiền xử lý (Kiểm tra xem load thành công chưa)
engineer.load_and_clean_base_data()
display(engineer.departures.head(10))

[*] 1/7: Đang làm sạch dữ liệu nền từ các file log Audit...
[V] Đã nạp 74723 chuyến cất cánh và 73797 chuyến hạ cánh.


,Crawl_Date,Scheduled_Time,Actual_Time,Destination,IATA,Airline,Flight_No,Terminal,Departure_Runway,Status,...,Is_Remote_Stand,Mode,Tail_Number,Origin,Destination_IATA,Origin_IATA,Event_Time,Actual_Event_Time,Scheduled_Event_Time,Departure_Delay
0,2025-12-15,2025-12-15 22:20:00,2025-12-15 22:32:00,Hanoi,HAN,Vietnam Airlines,VN7206,3.0,25L,Departed,...,NaN,departure,VN-A331,SGN,HAN,SGN,2025-12-15 22:20:00,2025-12-15 22:32:00,2025-12-15 22:20:00,12.0
1,2025-12-15,2025-12-15 22:05:00,2025-12-15 22:41:00,Hanoi,HAN,Vietnam Airlines,VN7268,3.0,25L,Departed,...,NaN,departure,VN-A339,SGN,HAN,SGN,2025-12-15 22:05:00,2025-12-15 22:41:00,2025-12-15 22:05:00,36.0
2,2025-12-15,2025-12-15 22:40:00,2025-12-15 23:45:00,Hanoi,HAN,VietJet Air,VJ522,1.0,25L,Departed,...,NaN,departure,VN-A637,SGN,HAN,SGN,2025-12-15 22:40:00,2025-12-15 23:45:00,2025-12-15 22:40:00,65.0
3,2025-12-16,2025-12-15 23:55:00,2025-12-16 00:04:00,Dubai,DXB,Emirates,EK393,2.0,25L,Departed,...,NaN,departure,A6-ENB,SGN,DXB,SGN,2025-12-15 23:55:00,2025-12-16 00:04:00,2025-12-15 23:55:00,9.0
4,2025-12-16,2025-12-15 23:45:00,2025-12-16 00:08:00,Tokyo,HND,Japan Airlines,JL70,2.0,25L,Departed,...,NaN,departure,JA829J,SGN,HND,SGN,2025-12-15 23:45:00,2025-12-16 00:08:00,2025-12-15 23:45:00,23.0
5,2025-12-16,2025-12-15 23:40:00,2025-12-16 00:13:00,Munich,MUC,Vietnam Airlines,VN33,2.0,25L,Departed,...,NaN,departure,VN-A862,SGN,MUC,SGN,2025-12-15 23:40:00,2025-12-16 00:13:00,2025-12-15 23:40:00,33.0
6,2025-12-16,2025-12-15 23:55:00,2025-12-16 00:19:00,Tokyo,NRT,VietJet Air,VJ822,2.0,25L,Departed,...,NaN,departure,VN-A607,SGN,NRT,SGN,2025-12-15 23:55:00,2025-12-16 00:19:00,2025-12-15 23:55:00,24.0
7,2025-12-16,2025-12-16 00:25:00,2025-12-16 00:21:00,Shanghai,PVG,Juneyao Air,HO1328,2.0,25L,Departed,...,NaN,departure,B-8408,SGN,PVG,SGN,2025-12-16 00:25:00,2025-12-16 00:21:00,2025-12-16 00:25:00,-4.0
8,2025-12-16,2025-12-15 23:50:00,2025-12-16 00:28:00,Seoul,ICN,Korean Air,KE476,2.0,25L,Departed,...,NaN,departure,HL8217,SGN,ICN,SGN,2025-12-15 23:50:00,2025-12-16 00:28:00,2025-12-15 23:50:00,38.0
9,2025-12-16,2025-12-16 00:20:00,2025-12-16 00:34:00,London,LHR,Vietnam Airlines,VN51,2.0,25L,Departed,...,NaN,departure,VN-A863,SGN,LHR,SGN,2025-12-16 00:20:00,2025-12-16 00:34:00,2025-12-16 00:20:00,14.0


### Cơ chế Xử lý Khuyết Giữa Arrival và Departure và Đồng bộ Chuỗi Xoay Vòng (Tail Rotation)
Máy bay vận hành theo chuỗi tịnh tiến: Hạ cánh (Arrival) -> Quay đầu (Turnaround) -> Cất cánh (Departure)
*  Tuy nhiên, dữ liệu cào (crawl) thực tế bị khuyết khoảng 1.5% số chuyến (Ví dụ: tàu bay bay charter không có lịch cố định).
*  Giải pháp toán học trong code:
1. Hàm pd.merge_asof(..., direction="backward") sẽ tự động tìm kiếm chặng hạ cánh gần nhất của chính cái đuôi tàu bay đó (Tail_Number) trong quá khứ để làm điểm tựa thời gian.
2. Với các chuyến bị missing (đã được đánh cờ Is_Imputed_Link = 1 thông qua file Audit), hệ thống không xóa bỏ mà tự động tính toán thời gian quay đầu trung vị (Median_Turnaround_Time) của phân khúc tàu bay đó tại sân bay đó làm thời gian đệm giả định.
3. Nhờ vậy, dòng thời gian thực tế chặng trước được bắc cầu sang chặng sau một cách hoàn hảo, giúp biến tích lũy trễ Accumulated_Delay không bị ngắt quãng.

In [47]:
# BƯỚC 2: Xử lý chuỗi xoay vòng (Khắc phục lỗi đứt link và tính delay)
engineer.build_tail_rotation_features()
display(engineer.features[['Flight_No', 'Scheduled_Time', 'Actual_Time', 'Turnaround_Buffer']].head(10))

[*] 2/7: Đang xây dựng chuỗi xoay vòng tàu bay (Tail Rotation)...
[V] Xử lý thành công dữ liệu xoay vòng.


,Flight_No,Scheduled_Time,Actual_Time,Turnaround_Buffer
0,QR975,2026-03-14 08:55:00,NaT,NaN
1,VJ3948,2026-03-14 18:05:00,2026-03-14 18:27:00,NaN
2,VJ374,2026-03-15 14:10:00,2026-03-15 14:46:00,NaN
3,7L8812,2026-03-09 08:00:00,2026-03-09 09:07:00,NaN
4,OK8902,2026-02-07 09:37:00,2026-02-07 06:45:00,5.0
5,OK8902,2026-02-10 11:51:00,2026-02-10 11:51:00,96.0
6,OK8903,2026-02-24 06:30:00,2026-02-24 06:30:00,19970.0
7,OK8903,2026-02-26 12:33:00,2026-02-26 12:33:00,23213.0
8,OK8903,2026-03-05 07:18:00,2026-03-05 07:18:00,32978.0
9,VJ515,2026-02-04 16:25:00,2026-02-04 17:03:00,NaN


### Rolling Window bằng Vector hóa
*  Thông thường, để tính mật độ sân bay trong khung giờ 30 phút hoặc 60 phút qua, chúng ta hay dùng hàm .rolling() của Pandas. Tuy nhiên, cách này bắt buộc phải tạo Index thời gian rất nặng và chạy vòng lặp cực kỳ chậm.
* Giải pháp tối ưu hóa:
	1.	Mã nguồn chuyển toàn bộ trục thời gian sang mảng một chiều của Numpy dưới định dạng số nguyên nano giây (datetime64[ns]) và tiến hành sắp xếp tăng dần (np.sort).
	2.	Hàm _count_between sử dụng thuật toán Tìm kiếm nhị phân (np.searchsorted) để xác định ngay lập tức vị trí con trỏ đầu và con trỏ cuối của mốc thời gian (Ví dụ: Vị trí của ‭$T - 30$‬‭‬ phút và ‭$T + 30$‬‭‬ phút).
	3. Số lượng chuyến bay trong khung giờ đó đơn giản là phép trừ vị trí: right_index - left_index (Độ phức tạp thuật toán giảm từ ‭$\mathcal{O}(N^2)$‬‭‬‭‬‭‬ xuống ‭$\mathcal{O}(N \log N)$‬‭‬‭‬‭‬‭‬‭‬). Phương pháp này giúp Notebook chạy nhanh hơn gấp 50 lần so với lệnh apply truyền thống.
* Gán nhãn Ngày Đặc biệt (Special Days)
Trường Is_Special_Days được gán giá trị = 1 khi ngày diễn ra hoạt động bay rơi vào các dịp cao điểm:
    * Kỳ nghỉ Giáng sinh: Ngày 24 tháng 12 và Ngày 25 tháng 12.
    * Năm mới (Tết Dương lịch): Ngày 31 tháng 12 và Ngày 1 tháng 1.
    * Giai đoạn cao điểm vận hành Tết Nguyên Đán (Việt Nam).
    * Ngày Giỗ Tổ Hùng Vương.
    * Ngày Giải phóng và Quốc tế Lao động: Ngày 30 tháng 4 và Ngày 1 tháng 5.
    * Ngày Quốc khánh: Ngày 2 tháng 9.
    * Cao điểm du lịch Hè: Toàn bộ các ngày trong Tháng 6 và Tháng 7.

In [48]:
# BƯỚC 3 & 4: Tính toán mật độ sân bay (Dùng hàm search nhị phân cực nhanh) và các biến Vận hành
engineer.build_airport_infrastructure_features()
engineer.build_operational_and_holiday_features()
display(engineer.features[['Flight_No', 'Airport_Load_Factor', 'Is_Wide_Body', 'Time_of_Day']].head(10))

[*] 3/7: Đang trích xuất đặc trưng Hạ tầng và Sự cố (Rolling Window)...
[V] Đã xử lý xong biến hạ tầng (np.searchsorted siêu tốc).
[*] 4/7: Đang lập lịch đặc trưng Vận hành và Lễ Tết...


,Flight_No,Airport_Load_Factor,Is_Wide_Body,Time_of_Day
0,QR975,43,0,Morning
1,VJ3948,45,0,Night
2,VJ374,37,0,Afternoon
3,7L8812,29,0,Morning
4,OK8902,28,0,Morning
5,OK8902,56,0,Morning
6,OK8903,26,0,Morning
7,OK8903,40,0,Afternoon
8,OK8903,25,0,Morning
9,VJ515,20,0,Afternoon


In [49]:
# BƯỚC 5 & 6 & 7: Hoàn thiện dữ liệu, Kiểm định và Xuất file
engineer.build_arrival_feature_rows()
engineer.evaluate_features_quality()
engineer.export_gold_outputs()
print("Toàn bộ luồng dữ liệu đã chạy trơn tru.")

[*] 5/7: Đang map feature tích lũy vào các chuyến hạ cánh...
[*] 6/7: Đang kiểm định chất lượng (Data Leakage & Imputation)...
[V] Xác thực Temporal Leakage thành công: 0 vi phạm.
[*] 7/7: Đang xuất bản dữ liệu sạch sang Gold Layer...
[V] TIẾN TRÌNH HOÀN TẤT! Dữ liệu Master được lưu tại: /Users/nguyenhung/PycharmProjects/DS108_AeroDelay/Data crawl/Gold_layer/Features/master_aero_features_gold.csv
Toàn bộ luồng dữ liệu đã chạy trơn tru.
